# 06 — Feature Engineering, Multicollinearity (VIF) & Feature Selection
**Project:** Axlero Solutions — Credit Risk Scoring and Loan Default Analysis  
**Module:** Feature Engineering & Selection (Module 4 — Pavani)

## 1. Objectives
1. Engineer 14 domain-specific financial and borrower features (exact loan-to-income ratio, 0.85 discrepancy flag, log income/loan amount, estimated annual debt service, debt-service-to-income, log residual income, employment-to-working-age ratio, credit-history-to-working-age ratio, ordinal grade, high burden flag, subprime grade flag, grade×burden interaction, and interest rate spread over grade median).
2. Fit all stateful parameters (imputation, 99.5th percentile income cap, age/employment caps, standard scalers) **strictly on `df_train`**.
3. Audit multicollinearity via Variance Inflation Factor (VIF) and rank features using a multi-criteria statistical + model-based composite score.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_raw_data, stratified_train_val_test_split
from src.preprocessing.cleaner import deduplicate_and_invalidate
from src.features.feature_engineering import (
    CreditRiskFeatureEngineer,
    ENGINEERED_FEATURE_DOCS,
    compute_vif_table,
    rank_and_select_features,
)

print("Engineered Domain Features Specification:")
display(pd.DataFrame(ENGINEERED_FEATURE_DOCS))

Engineered Domain Features Specification:


,feature,formula,business_relevance
0,calc_loan_to_income,loan_amnt / person_income,"Exact debt-to-income ratio removing the 1,005 ..."
1,ratio_discrepancy_flag,1 if |loan_amnt / person_income - loan_percent...,Identifies the 387 records where person_income...
2,log_person_income,ln(1 + person_income),Compresses extreme right skewness in annual in...
3,log_loan_amnt,ln(1 + loan_amnt),Normalizes loan principal distribution across ...
4,est_annual_debt_service,loan_amnt * (1 + loan_int_rate / 100) / 3.0,Estimates annual principal + interest repaymen...
5,debt_service_to_income,est_annual_debt_service / person_income,Captures total interest-adjusted payment shock...
6,log_residual_income,"ln(1 + max(0, person_income - est_annual_debt_...",Measures the post-debt-service disposable inco...
7,emp_to_working_age_ratio,"person_emp_length / max(1, person_age - 18)",Measures career tenure stability relative to t...
8,cred_hist_to_working_age_ratio,"cb_person_cred_hist_length / max(1, person_age...",Measures credit file maturity relative to adul...
9,loan_grade_num,"Ordinal mapping A=1, B=2, C=3, D=4, E=5, F=6, G=7",Preserves monotonic credit risk rank order acr...


In [2]:
df_interim, _ = deduplicate_and_invalidate(load_raw_data())
df_train, df_val, df_test = stratified_train_val_test_split(df_interim, random_state=42)

fe = CreditRiskFeatureEngineer(imputation_strategy="grade_median", scale_numeric=True)
fe.fit(df_train.drop(columns=["loan_status"]), df_train["loan_status"])

X_train_full = fe.transform(df_train.drop(columns=["loan_status"]))
y_train = df_train["loan_status"].astype(int)

ranking_df, selected_features = rank_and_select_features(X_train_full, y_train, random_state=42)
print(f"Selected {len(selected_features)} of {X_train_full.shape[1]} features:")
display(ranking_df)

Selected 28 of 30 features:


,feature,anova_f_stat,anova_p_value,mutual_info,l1_abs_coef,rf_gini_importance,composite_score,rank,selected
0,grade_x_burden,6301.02,0.000000e+00,0.1222,0.3282,0.0656,0.6549,1,True
1,calc_loan_to_income,3954.03,0.000000e+00,0.1083,0.3349,0.0892,0.6202,2,True
2,debt_service_to_income,4278.07,0.000000e+00,0.1067,0.0000,0.0807,0.5862,3,True
3,high_burden_flag,5082.32,0.000000e+00,0.0729,2.5315,0.0468,0.5703,4,True
4,subprime_grade_flag,4263.28,0.000000e+00,0.0682,1.6501,0.0720,0.5636,5,True
5,loan_grade_num,3600.18,0.000000e+00,0.0690,0.7009,0.0842,0.5323,6,True
6,home_RENT,1371.79,1.328580e-291,0.0283,2.4533,0.1043,0.5228,7,True
7,loan_percent_income,3794.22,0.000000e+00,0.0837,0.0000,0.0611,0.4674,8,True
8,log_person_income,1939.11,0.000000e+00,0.1106,0.5924,0.0363,0.4215,9,True
9,log_residual_income,2282.68,0.000000e+00,0.0786,0.7183,0.0482,0.4074,10,True


## 2. Multicollinearity Audit (Variance Inflation Factor — VIF)

In [3]:
vif_selected = compute_vif_table(X_train_full[selected_features])
print("VIF Table on Selected Feature Set:")
display(vif_selected)

VIF Table on Selected Feature Set:


,feature,vif,multicollinearity_status
0,log_residual_income,473626.10,"High (>= 10, Collinear)"
1,log_person_income,442069.66,"High (>= 10, Collinear)"
2,debt_service_to_income,12631.99,"High (>= 10, Collinear)"
3,calc_loan_to_income,11108.24,"High (>= 10, Collinear)"
4,loan_percent_income,1032.65,"High (>= 10, Collinear)"
5,loan_grade_num,102.26,"High (>= 10, Collinear)"
6,loan_int_rate,87.44,"High (>= 10, Collinear)"
7,grade_x_burden,66.74,"High (>= 10, Collinear)"
8,log_loan_amnt,20.14,"High (>= 10, Collinear)"
9,cb_person_cred_hist_length,15.63,"High (>= 10, Collinear)"
